In [2]:
import os
import requests
import pandas as pd
from datetime import datetime
import pytz

# === Timezone Setup ===
eastern = pytz.timezone('US/Eastern')
today = datetime.now(eastern).date()
today_str = today.isoformat()

# === Helpers for safe conversion ===
def safe_float(x):
    try:
        return float(x)
    except (ValueError, TypeError):
        return 0.0

def safe_int(x):
    try:
        return int(x)
    except (ValueError, TypeError):
        return 0

# === Fetch Pitcher Stats ===
def get_pitcher_stats(player_id, season):
    url = f"https://statsapi.mlb.com/api/v1/people/{player_id}/stats"
    params = {
        "stats": "season",
        "season": season,
        "group": "pitching",
        "gameType": "R"
    }
    try:
        r = requests.get(url, params=params)
        r.raise_for_status()
        data = r.json()
        splits = data.get("stats", [])[0].get("splits", [])
        stat = splits[0].get("stat", {}) if splits else {}
        return {
            "ERA": safe_float(stat.get("era")),
            "WHIP": safe_float(stat.get("whip")),
            "W": safe_int(stat.get("wins")),
            "SO": safe_int(stat.get("strikeOuts")),
            "IP": safe_float(stat.get("inningsPitched")),
            "AVG": stat.get("avg"),
            "AB": safe_int(stat.get("atBats"))
        }
    except:
        return {}

# === Fetch Team Pitching Stats ===
def get_team_pitching_stats(team_id, season):
    url = f"https://statsapi.mlb.com/api/v1/teams/{team_id}/stats"
    params = {
        "stats": "season",
        "season": season,
        "group": "pitching"
    }
    try:
        r = requests.get(url, params=params)
        r.raise_for_status()
        data = r.json()
        splits = data.get("stats", [])[0].get("splits", [])
        stat = splits[0].get("stat", {}) if splits else {}
        return {
            "ERA": safe_float(stat.get("era")),
            "WHIP": safe_float(stat.get("whip")),
            "SO": safe_int(stat.get("strikeOuts")),
            "IP": safe_float(stat.get("inningsPitched")),
            "AVG": stat.get("avg"),
            "AB": safe_int(stat.get("atBats"))
        }
    except:
        return {}

# === Estimate Bullpen Stats ===
def estimate_bullpen_stats(team_stats, starter_stats):
    team_ip = team_stats.get("IP", 0.0)
    starter_ip = starter_stats.get("IP", 0.0)
    bullpen_ip = max(team_ip - starter_ip, 0.1)

    def diff(metric):
        return max(team_stats.get(metric, 0) - starter_stats.get(metric, 0), 0)

    return {
        "ERA": round((team_stats.get("ERA", 0.0) * team_ip - starter_stats.get("ERA", 0.0) * starter_ip) / bullpen_ip, 2),
        "WHIP": round((team_stats.get("WHIP", 0.0) * team_ip - starter_stats.get("WHIP", 0.0) * starter_ip) / bullpen_ip, 2),
        "SO": diff("SO"),
        "IP": round(bullpen_ip, 1),
        "AVG": None,
        "AB": diff("AB")
    }

# === Main Data Gathering ===
def fetch_starting_pitchers_with_bullpens(target_date):
    date_str = target_date.isoformat()
    url = f'https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={date_str}&hydrate=probablePitcher(note,stats,person)'

    r = requests.get(url)
    if r.status_code != 200:
        print(f"❌ API request failed: {r.status_code}")
        return None

    games = r.json().get('dates', [])[0].get('games', []) if r.json().get('dates') else []
    rows = []

    for game in games:
        home = game['teams']['home']
        away = game['teams']['away']

        home_team = home['team']['name']
        home_id = home['team']['id']
        home_pitcher = home.get('probablePitcher', {})
        home_name = home_pitcher.get('fullName', 'TBD')
        home_pid = home_pitcher.get('id')

        away_team = away['team']['name']
        away_id = away['team']['id']
        away_pitcher = away.get('probablePitcher', {})
        away_name = away_pitcher.get('fullName', 'TBD')
        away_pid = away_pitcher.get('id')

        # Fetch stats
        home_sp = get_pitcher_stats(home_pid, today.year) if home_pid else {}
        away_sp = get_pitcher_stats(away_pid, today.year) if away_pid else {}

        home_team_stats = get_team_pitching_stats(home_id, today.year)
        away_team_stats = get_team_pitching_stats(away_id, today.year)

        home_bp = estimate_bullpen_stats(home_team_stats, home_sp) if home_team_stats and home_sp else {}
        away_bp = estimate_bullpen_stats(away_team_stats, away_sp) if away_team_stats and away_sp else {}

        rows.append({
            "Date": date_str,
            "Home Team": home_team,
            "Home Starter": home_name,
            "Home ERA": home_sp.get("ERA"),
            "Home WHIP": home_sp.get("WHIP"),
            "Home IP": home_sp.get("IP"),
            "Home SO": home_sp.get("SO"),
            "Home AVG": home_sp.get("AVG"),
            "Home AB": home_sp.get("AB"),
            "Home BP ERA": home_bp.get("ERA"),
            "Home BP WHIP": home_bp.get("WHIP"),
            "Home BP IP": home_bp.get("IP"),
            "Home BP SO": home_bp.get("SO"),
            "Home BP AB": home_bp.get("AB"),

            "Away Team": away_team,
            "Away Starter": away_name,
            "Away ERA": away_sp.get("ERA"),
            "Away WHIP": away_sp.get("WHIP"),
            "Away IP": away_sp.get("IP"),
            "Away SO": away_sp.get("SO"),
            "Away AVG": away_sp.get("AVG"),
            "Away AB": away_sp.get("AB"),
            "Away BP ERA": away_bp.get("ERA"),
            "Away BP WHIP": away_bp.get("WHIP"),
            "Away BP IP": away_bp.get("IP"),
            "Away BP SO": away_bp.get("SO"),
            "Away BP AB": away_bp.get("AB")
        })

    return pd.DataFrame(rows)

# === Execute and Save ===
df = fetch_starting_pitchers_with_bullpens(today)

if df is not None and not df.empty:
    pd.set_option("display.max_columns", None)
    print("✅ Starting pitchers and bullpen stats:")
    print(df.head())

    output_dir = os.path.abspath(os.path.join("..", "data", "starting-pitchers"))
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, f"starting_pitchers_{today_str}.csv")
    df.to_csv(output_file, index=False)
    print(f"✅ CSV saved to {output_file}")
else:
    print("⚠️ No data found.")


✅ Starting pitchers and bullpen stats:
         Date             Home Team     Home Starter  Home ERA  Home WHIP  \
0  2025-06-19       Cincinnati Reds    Nick Martinez      4.55       1.28   
1  2025-06-19  Washington Nationals  Trevor Williams      5.54       1.45   
2  2025-06-19      New York Yankees     Carlos Rodón      3.10       0.98   
3  2025-06-19        Detroit Tigers     Tarik Skubal      2.06       0.85   
4  2025-06-19        Detroit Tigers     Tyler Holton      4.72       1.31   

   Home IP  Home SO Home AVG  Home AB  Home BP ERA  Home BP WHIP  Home BP IP  \
0     83.0       62     .274      318         3.82          1.21       578.0   
1     74.2       58     .297      303         4.83          1.37       587.8   
2     95.2      114     .178      342         3.59          1.21       561.0   
3     96.0      117     .202      347         3.57          1.25       581.1   
4     34.1       30     .280      132         3.29          1.18       643.0   

   Home BP SO  Ho